# 10 — Multi-Task PPO: Train on 1-1 and 1-2 Simultaneously

Trains a single PPO agent on **both** World 1-1 and World 1-2 at the same time,
avoiding catastrophic forgetting.

**Strategy:**
- Start from the pre-trained 1-1 model (already knows 1-1)
- 8 parallel envs: **2 on 1-1** (retention) + **6 on 1-2** (learning)
- More 1-2 envs so the harder level dominates the gradient
- Higher entropy (0.02) to encourage exploration on the new level
- The 1-1 envs act as "regularization" against catastrophic forgetting

**Comparison:**
- Single-task 1-1 → wins 1-1, fails 1-2
- Transfer 1-1→1-2 → wins 1-2, forgets 1-1 (catastrophic forgetting)
- **Multi-task** → should win both levels

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/contente/Documents/ENSTA/autonomous/CSC-52081-ContinousMountainCar


In [2]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_symbolic_multitask_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback

print(f'Using CUDA: {torch.cuda.is_available()}')

Using CUDA: True


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
# Create 8 parallel envs: 2 on World 1-1 (retention), 6 on World 1-2 (learning)
LEVELS = ['SuperMarioBros-1-1-v3', 'SuperMarioBros-1-2-v3']
ENVS_PER_LEVEL = [2, 6]  # asymmetric: more weight on the harder level

env = make_symbolic_multitask_vec_env(
    env_ids=LEVELS,
    skip=4,
    n_stack=4,
    flatten=True,
    envs_per_level=ENVS_PER_LEVEL,
)

total_envs = sum(ENVS_PER_LEVEL)
print(f'Levels: {LEVELS}')
print(f'Envs per level: {ENVS_PER_LEVEL}')
print(f'Total parallel envs: {total_envs}')
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: como faz para ver o mario{env.action_space.n}')

In [ ]:
# Load pre-trained 1-1 model and continue training on both levels
model = PPO.load(
    '../models/symbolic_ppo/final_model',
    env=env,
    device='cpu',
    learning_rate=2.5e-4,
    ent_coef=0.02,  # higher entropy to encourage exploration on 1-2
)

TOTAL_STEPS = 4_000_000

print(f'Loaded pre-trained 1-1 model')
print(f'Training: {TOTAL_STEPS:,} steps on both levels')
print(f'Device: {model.device}')
print(f'Entropy coef: {model.ent_coef} (higher for exploration)')
print(f'Batch per rollout: {512 * total_envs} samples')

In [5]:
%load_ext tensorboard
%tensorboard --logdir ../logs/symbolic_ppo_multitask

In [6]:
# Train
callback = CheckpointAndLogCallback(
    save_path='../models/symbolic_ppo_multitask',
    save_freq=100_000,
)

model.learn(
    total_timesteps=TOTAL_STEPS,
    callback=callback,
    log_interval=1,
)
model.save('../models/symbolic_ppo_multitask/final_model')
print('Multi-task training complete!')

Logging to ../logs/symbolic_ppo_multitask/PPO_1


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional infor

-----------------------------
| time/              |      |
|    fps             | 252  |
|    iterations      | 1    |
|    time_elapsed    | 16   |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 251         |
|    iterations           | 2           |
|    time_elapsed         | 32          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.011135597 |
|    clip_fraction        | 0.118       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.94       |
|    explained_variance   | -0.0139     |
|    learning_rate        | 0.00025     |
|    loss                 | 0.715       |
|    n_updates            | 4           |
|    policy_gradient_loss | -0.0128     |
|    value_loss           | 1.88        |
-----------------------------------------
----------------------------------

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| time/                   |             |
|    fps                  | 263         |
|    iterations           | 29          |
|    time_elapsed         | 450         |
|    total_timesteps      | 118784      |
| train/                  |             |
|    approx_kl            | 0.013815174 |
|    clip_fraction        | 0.182       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.32       |
|    explained_variance   | 0.653       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.864       |
|    n_updates            | 112         |
|    policy_gradient_loss | -0.0122     |
|    value_loss           | 2.38        |
-----------------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 263         |
|    iterations           | 30          |
|    time_elapsed         | 465         |
|    total_timesteps      | 122880

...

-----------------------------------------
| time/                   |             |
|    fps                  | 265         |
|    iterations           | 39          |
|    time_elapsed         | 601         |
|    total_timesteps      | 159744      |
| train/                  |             |
|    approx_kl            | 0.015216933 |
|    clip_fraction        | 0.183       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.17       |
|    explained_variance   | 0.774       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.978       |
|    n_updates            | 152         |
|    policy_gradient_loss | -0.0165     |
|    value_loss           | 2.13        |
-----------------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 265         |
|    iterations           | 40          |
|    time_elapsed         | 617         |
|    total_timesteps      | 163840

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| time/                   |             |
|    fps                  | 265         |
|    iterations           | 52          |
|    time_elapsed         | 802         |
|    total_timesteps      | 212992      |
| train/                  |             |
|    approx_kl            | 0.028943814 |
|    clip_fraction        | 0.26        |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.24       |
|    explained_variance   | 0.892       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.37        |
|    n_updates            | 204         |
|    policy_gradient_loss | -0.0117     |
|    value_loss           | 1.24        |
-----------------------------------------


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| time/                   |             |
|    fps                  | 265         |
|    iterations           | 53          |
|    time_elapsed         | 818         |
|    total_timesteps      | 217088      |
| train/                  |             |
|    approx_kl            | 0.026159491 |
|    clip_fraction        | 0.251       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.28       |
|    explained_variance   | 0.926       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.536       |
|    n_updates            | 208         |
|    policy_gradient_loss | -0.00913    |
|    value_loss           | 0.926       |
-----------------------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 265        |
|    iterations           | 54         |
|    time_elapsed         | 833        |
|    total_timesteps      | 221184     

## Evaluate on each level separately

In [7]:
import numpy as np
from stable_baselines3 import PPO

eval_model = PPO.load('../models/symbolic_ppo_multitask/final_model')

for level in LEVELS:
    print(f'\n{"="*50}')
    print(f'Evaluating on: {level}')
    print(f'{"="*50}')

    eval_env = make_symbolic_env(
        env_id=level, skip=4, n_stack=4, flatten=True,
    )

    rewards, flags = [], []
    for ep in range(10):
        obs = eval_env.reset()
        obs = obs[0] if isinstance(obs, tuple) else obs
        done, total_reward, flag = False, 0.0, False

        while not done:
            action, _ = eval_model.predict(obs, deterministic=True)
            result = eval_env.step(int(action))
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
            total_reward += float(reward)
            if isinstance(info, dict) and info.get('flag_get', False):
                flag = True

        rewards.append(total_reward)
        flags.append(flag)
        status = 'FLAG!' if flag else 'DEAD'
        print(f'  Ep {ep+1}: reward={total_reward:.1f} [{status}]')

    print(f'\n  Mean reward: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
    print(f'  Flag rate: {np.mean(flags):.0%}')
    eval_env.close()

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(



Evaluating on: SuperMarioBros-1-1-v3


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(


  Ep 1: reward=315.1 [FLAG!]
  Ep 2: reward=315.1 [FLAG!]
  Ep 3: reward=315.1 [FLAG!]
  Ep 4: reward=315.1 [FLAG!]
  Ep 5: reward=315.1 [FLAG!]
  Ep 6: reward=315.1 [FLAG!]
  Ep 7: reward=315.1 [FLAG!]
  Ep 8: reward=315.1 [FLAG!]
  Ep 9: reward=315.1 [FLAG!]
  Ep 10: reward=315.1 [FLAG!]

  Mean reward: 315.1 +/- 0.0
  Flag rate: 100%

Evaluating on: SuperMarioBros-1-2-v3
  Ep 1: reward=51.2 [DEAD]
  Ep 2: reward=51.2 [DEAD]
  Ep 3: reward=51.2 [DEAD]
  Ep 4: reward=51.2 [DEAD]
  Ep 5: reward=51.2 [DEAD]
  Ep 6: reward=51.2 [DEAD]
  Ep 7: reward=51.2 [DEAD]
  Ep 8: reward=51.2 [DEAD]
  Ep 9: reward=51.2 [DEAD]
  Ep 10: reward=51.2 [DEAD]

  Mean reward: 51.2 +/- 0.0
  Flag rate: 0%


In [8]:
# Watch: python watch_agent.py --model ../models/symbolic_ppo_multitask/final_model --env SuperMarioBros-1-1-v3
# Watch: python watch_agent.py --model ../models/symbolic_ppo_multitask/final_model --env SuperMarioBros-1-2-v3